In [ ]:
import numpy as np
from scipy.integrate import trapezoid
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy.optimize import brentq
from scipy.stats import skew, kurtosis
from scipy.signal import welch, hilbert

In [ ]:
#parameters
Hs = 6.0       # Significant wave height [m]
g = 9.81       # Gravitational acceleration [m/s^2]
s0p = 0.05     # Chosen deep-water wave steepness Hs / lambda_0p [-] (young wind waves, Schiereck & Verhagen p. 180)

# Calculate peak period Tp [s]
Tp = np.sqrt((2 * np.pi * Hs) / (g * s0p))

# Equivalent dimensionless steepness ka = pi * s0p (Lecture 3, Slide 3 & 9)
ka = np.pi * s0p

print(f"Peak period Tp = {Tp:.2f} s")
print(f"Steepness ka   = {ka:.3f}")

In [ ]:
fp = 1.0 / Tp   # Peak frequency [Hz]

# JONSWAP parameters (Lecture 5, slide 12)
gamma = 3.3
sigma_a = 0.07  # for f <= fp
sigma_b = 0.09  # for f > fp

# 2. Frequency grid (avoid f = 0 to prevent division by zero)
f = np.linspace(0.01, 1.0, 10000)
sigma = np.where(f <= fp, sigma_a, sigma_b)

# 3. Pierson-Moskowitz spectrum E_PM(f)
E_PM = (5.0 / 16.0) * (Hs**2) * (fp**4) * (f**(-5)) * np.exp(-1.25 * (fp / f)**4)

# 4. JONSWAP spectrum E(f) with normalizing factor A_gamma
A_gamma = 1.0 - 0.287 * np.log(gamma)
peak_enhancement = gamma ** np.exp(-((f - fp)**2) / (2.0 * (sigma**2) * (fp**2)))
E = A_gamma * E_PM * peak_enhancement

# 5. Compute spectral moments
m_minus1 = trapezoid(f**(-1) * E, f)
m0 = trapezoid(E, f)
m1 = trapezoid(f**1 * E, f)
m2 = trapezoid(f**2 * E, f)

# 6. Compute characteristic wave periods
Tm_10 = m_minus1 / m0        # Energy period T_{m-1,0}
Tm01 = m0 / m1               # Mean spectral period T_{m01}
Tm02 = np.sqrt(m0 / m2)      # Mean zero-crossing period T_{m02} (T_z / T_m)

print(f"A_gamma         = {A_gamma:.5f}")
print(f"m0              = {m0:.4f} m^2  (target Hs^2/16 = {Hs**2 / 16:.4f} m^2)")
print(f"Check Hs (4√m0) = {4 * np.sqrt(m0):.3f} m")
print(f"T_m-1,0         = {Tm_10:.2f} s")
print(f"T_m01           = {Tm01:.2f} s")
print(f"T_m02 (T_m)     = {Tm02:.2f} s")

# 7. Plot both PM and JONSWAP spectra for comparison
plt.figure(figsize=(8, 4.5))
plt.plot(f, E_PM, color="#1b263b", linestyle="--", linewidth=1.8, label="PM (γ = 1.0)")
plt.plot(f, E, color="#00a6d6", linewidth=2.2, label=f"JONSWAP (γ = {gamma})")
plt.fill_between(f, E, alpha=0.15, color="#00a6d6")
plt.axvline(fp, color="#d62728", linestyle=":", linewidth=1.2, label=f"fp = {fp:.3f} Hz")

plt.xlim(0, 0.35)
plt.ylim(0, np.max(E) * 1.1)
plt.xlabel("Frequency f [Hz]")
plt.ylabel("Variance Density E(f) [m²/Hz]")
plt.title(f"JONSWAP vs. PM Spectrum (Hs = {Hs} m, Tp = {Tp} s)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Peak frequency fp = {fp:.3f} Hz")

In [ ]:
h = 20.0  # Water depth [m]

def local_wave_steepness(Hs, T, h):
    omega = 2 * np.pi / T
    # Solve omega^2 = g * k * tanh(k * h) using deep-water k0 as initial guess
    k = fsolve(lambda k: omega**2 - g * k * np.tanh(k * h), omega**2 / g)[0]
    L = 2 * np.pi / k
    return Hs / L, L, k

for name, T in [("Tp", Tp), ("T_m-1,0", Tm_10), ("T_m01", Tm01), ("T_m02", Tm02)]:
    s, L, k = local_wave_steepness(Hs, T, h)
    print(f"{name:7s} = {T:.2f} s  ->  L = {L:6.2f} m,  s = {s:.4f}")

In [ ]:
# NOTE: first draft of Q4a (seed 42, f up to 0.5 Hz, no normalisation).
# SUPERSEDED by the model in the next cell, which is used for Q4a-c, Q5 and Q6.
# Do not use the variables of this cell further on.

# 1. Wave and site parameters
Hs = 6.0        # Significant wave height [m]
Tp = 8.77      # Peak period [s]
fp = 1.0 / Tp   # Peak frequency [Hz]
gamma = 3.3     # JONSWAP peak enhancement factor
h = 20.0        # Water depth [m]
g = 9.81        # Gravity [m/s^2]
z = 0.0         # Evaluation elevation [m] (mean water level)

# 2. Time and frequency grids
D = 10800.0      # Record duration [s] (180 minutes)
dt = 0.25       # Time step [s]
t = np.arange(0, D, dt)

df = 1.0 / D    # Frequency resolution [Hz] (prevents repetition within D)
f = np.arange(0.02, 0.5, df)  # Frequency array [Hz] (avoid f = 0)
omega = 2 * np.pi * f

# 3. JONSWAP variance density spectrum E(f) (Lecture 5)
sigma = np.where(f <= fp, 0.07, 0.09)
A_gamma = 1.0 - 0.287 * np.log(gamma)
E_PM = (5.0 / 16.0) * (Hs**2) * (fp**4) * (f**(-5)) * np.exp(-1.25 * (fp / f)**4)
E = A_gamma * E_PM * (gamma ** np.exp(-((f - fp)**2) / (2.0 * (sigma**2) * (fp**2))))

# 4. Solve linear dispersion relation k(f) for every frequency component
k = np.array([
    fsolve(lambda ki: w**2 - g * ki * np.tanh(ki * h), w**2 / g)[0]
    for w in omega
])

# 5. Linear transfer function for horizontal velocity u at elevation z
H_u = omega * np.cosh(k * (z + h)) / np.sinh(k * h)

# 6. Random-phase synthesis
rng = np.random.default_rng(42)            # Fixed seed for reproducibility
a = np.sqrt(2.0 * E * df)                  # Component amplitudes [m]
alpha = rng.uniform(0, 2 * np.pi, len(f))  # Random phases in [0, 2*pi)

# Vectorized summation over all frequency components
phase = omega[:, None] * t[None, :] + alpha[:, None]
eta = np.sum(a[:, None] * np.cos(phase), axis=0)
u = np.sum((H_u * a)[:, None] * np.cos(phase), axis=0)

# 7. Sanity checks
print(f"eta(t):   std = {np.std(eta):.3f} m  |  4*std (Hs check) = {4*np.std(eta):.3f} m  |  max = {np.max(eta):.2f} m")
print(f"u(0, t):  u_rms = {np.std(u):.3f} m/s  |  max = {np.max(u):.2f} m/s")

# 8. Plot first 120 seconds of both time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t, eta, color="#00a6d6", linewidth=1.2)
ax1.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax1.set_ylabel(r"$\eta(t)$ [m]")
ax1.set_title(f"Linear Random-Phase Wave Field at z = {z} m (Hs = {Hs} m, Tp = {Tp} s, h = {h} m)")
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.plot(t, u, color="#e6550d", linewidth=1.2)
ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.set_xlabel("Time t [s]")
ax2.set_ylabel(r"$u(z=0, t)$ [m/s]")
ax2.set_xlim(0, 120)
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1. Shared Site & Sea-State Parameters (consistent with Q3)
# ============================================================
Hs = 6.0        # Significant wave height [m] (Table 2)
h = 20.0        # Water depth [m] (Table 2)
g = 9.81        # Gravitational acceleration [m/s^2]
s0p = 0.05      # Deep-water peak steepness [-] (Q3: young wind waves)
Tp = np.sqrt((2.0 * np.pi * Hs) / (g * s0p))  # Peak period [s] (= 8.77 s)
gamma = 3.3     # JONSWAP peak enhancement factor (Lecture 5, slide 12)

# Discretization parameters
D = 10800.0     # Record duration [s] (3 h, max. duration of a stationary sea state, Lecture 5 slide 4)
dt = 0.25       # Time step [s] (f_Nyq = 2.0 Hz, well above 2*f_max)
f_min = 0.02    # Lower cutoff frequency [Hz] (~0.18 fp)
f_max = 0.35    # Upper cutoff frequency [Hz] (~3.1 fp)


# ============================================================
# 2. Helper Functions (reusable for Parts I, II, and III)
# ============================================================
def solve_dispersion(omega, h, g=9.81):
    """Solve linear dispersion relation omega^2 = g*k*tanh(k*h) (Lecture 3)."""
    k = np.empty_like(omega, dtype=float)
    for idx, w in enumerate(omega):
        k_low = 0.5 * w / np.sqrt(g * h)
        k_high = max(2.0 * w**2 / g, 2.0 * w / np.sqrt(g * h)) + 1.0
        k[idx] = brentq(lambda ki: w**2 - g * ki * np.tanh(ki * h), k_low, k_high)
    return k


def jonswap_spectrum(f, Hs, Tp, gamma=3.3, normalize=True):
    """JONSWAP variance density spectrum E(f) [m^2/Hz] (Lecture 5, slide 12)."""
    fp = 1.0 / Tp
    sigma = np.where(f <= fp, 0.07, 0.09)
    A_gamma = 1.0 - 0.287 * np.log(gamma)
    E_PM = (5.0 / 16.0) * (Hs**2) * (fp**4) * (f**(-5)) * np.exp(-1.25 * (fp / f)**4)
    E = A_gamma * E_PM * (gamma ** np.exp(-((f - fp)**2) / (2.0 * (sigma**2) * (fp**2))))
    if normalize:
        # ASSUMPTION: rescale so that 4*sqrt(m0) == Hs despite the cutoffs [f_min, f_max]
        df_loc = f[1] - f[0]
        E *= (Hs**2 / 16.0) / np.sum(E * df_loc)
    return E


def build_linear_components(Hs, Tp, h, D=10800.0, dt=0.25, f_min=0.02, f_max=0.35,
                            gamma=3.3, g=9.81, seed=41):
    """Discrete frequency grid, amplitudes a_i = sqrt(2 E df), wavenumbers and random phases
    (Lecture 5, appendix: synthesising wave records)."""
    N_t = int(round(D / dt))
    df = 1.0 / D                       # record repeats after 1/df = D

    n_min = int(np.ceil(f_min / df))
    n_max = int(np.floor(f_max / df))
    assert 2 * n_max < N_t // 2, "Decrease dt: 2*f_max must stay below the Nyquist frequency"

    n_idx = np.arange(n_min, n_max + 1)  # integer harmonics -> exact FFT synthesis
    f = n_idx * df
    omega = 2.0 * np.pi * f
    k = solve_dispersion(omega, h, g)
    E = jonswap_spectrum(f, Hs, Tp, gamma=gamma, normalize=True)

    a = np.sqrt(2.0 * E * df)
    rng = np.random.default_rng(seed)
    alpha = rng.uniform(0.0, 2.0 * np.pi, len(f))

    return {"N_t": N_t, "dt": dt, "D": D, "t": np.arange(N_t) * dt,
            "df": df, "n_idx": n_idx, "f": f, "omega": omega,
            "k": k, "E": E, "a": a, "alpha": alpha, "h": h, "g": g}


def synthesize_linear(comp, z=0.0, x=0.0):
    """Q4a: linear eta(t), u(z,t) and du/dt(z,t) at (x, z) via exact FFT synthesis.
    eta = sum a_i cos(omega_i t - k_i x + alpha_i);  u transfer: omega cosh k(z+h)/sinh kh (Lecture 3, slide 26)."""
    N_t = comp["N_t"]
    n_idx, omega, k = comp["n_idx"], comp["omega"], comp["k"]
    a, alpha, h = comp["a"], comp["alpha"], comp["h"]

    C1 = a * np.exp(1j * (alpha - k * x))
    # Overflow-safe form of cosh(k(z+h))/sinh(kh), valid for -h <= z <= 0
    H_u = omega * (np.exp(k * z) + np.exp(-k * (z + 2.0 * h))) / (1.0 - np.exp(-2.0 * k * h))

    def to_time(coeff):
        X = np.zeros(N_t // 2 + 1, dtype=np.complex128)
        X[n_idx] = (N_t / 2.0) * coeff
        return np.fft.irfft(X, n=N_t)

    return {"t": comp["t"],
            "eta1": to_time(C1),
            "u1": to_time(H_u * C1),
            "dudt1": to_time(1j * omega * H_u * C1)}   # d/dt -> multiply by i*omega


def narrowband_bound_waves(eta1, Tp, h, z=0.0, g=9.81):
    """Q4b + Q4c: narrowband second-order bound waves from the envelope of eta1.
    Lecture 6, slide 17: analytic signal eta1 + i*H[eta1] = A exp(i*theta),
      super-harmonic eta22 = G(kp h) kp A^2 cos(2 theta),
      G(kh) = cosh kh (2 + cosh 2kh) / (4 sinh^3 kh)   (Lecture 6, slide 6).
    u22 from the 2nd-order potential (Lecture 6, slide 6 / appendix slide 2), differentiated to x.
    Lecture 6, slide 19: sub-harmonic eta = -Sxx / (rho (gh - cg^2)), Sxx = E (2n - 1/2) (Lecture 7),
      with E = 1/2 rho g A^2 and the mean removed (zero-mean record).
    The FFT-synthesised record is exactly periodic, so the Hilbert transform has no edge effects."""
    z_a = hilbert(eta1)
    A = np.abs(z_a)                 # envelope A(t)
    theta = np.angle(z_a)           # phase theta(t)

    omega_p = 2.0 * np.pi / Tp
    k_p = solve_dispersion(np.array([omega_p]), h, g)[0]
    kph = k_p * h
    G = np.cosh(kph) * (2.0 + np.cosh(2.0 * kph)) / (4.0 * np.sinh(kph)**3)

    eta22 = G * k_p * A**2 * np.cos(2.0 * theta)
    u22 = 0.75 * omega_p * k_p * A**2 * np.cosh(2.0 * k_p * (z + h)) / np.sinh(kph)**4 * np.cos(2.0 * theta)

    n = 0.5 * (1.0 + 2.0 * kph / np.sinh(2.0 * kph))
    cg = n * omega_p / k_p
    eta_sub = -(2.0 * n - 0.5) * (A**2 - np.mean(A**2)) / (2.0 * (h - cg**2 / g))

    return {"A": A, "theta": theta, "eta22": eta22, "u22": u22, "eta_sub": eta_sub,
            "k_p": k_p, "kph": kph, "G": G, "n": n, "cg": cg}


# ============================================================
# 3. Run Simulation at (x = 0, z = 0)
# ============================================================
comp = build_linear_components(Hs, Tp, h, D=D, dt=dt, f_min=f_min, f_max=f_max, gamma=gamma, seed=41)
lin = synthesize_linear(comp, z=0.0, x=0.0)
t, eta1, u1 = lin["t"], lin["eta1"], lin["u1"]

bw = narrowband_bound_waves(eta1, Tp, h, z=0.0)
eta22, u22, eta_sub = bw["eta22"], bw["u22"], bw["eta_sub"]
eta = eta1 + eta22          # Q4b: linear + super-harmonic
u = u1 + u22

print(f"Sea state: Hs = {Hs:.2f} m, Tp = {Tp:.2f} s (s0p = {s0p}), h = {h:.1f} m, N_f = {len(comp['f'])}")
print(f"Carrier: kp*h = {bw['kph']:.3f}, G = {bw['G']:.3f}, cg/sqrt(gh) = {bw['cg']/np.sqrt(g*h):.2f}")
print(f"Linear eta1 : 4*std = {4*np.std(eta1):.3f} m | max = {np.max(eta1):+.2f} m | min = {np.min(eta1):+.2f} m | skew = {skew(eta1):+.3f}")
print(f"Super eta22 :   std = {np.std(eta22):.3f} m | max = {np.max(eta22):+.2f} m | min = {np.min(eta22):+.2f} m")
print(f"Total  eta  : 4*std = {4*np.std(eta):.3f} m | max = {np.max(eta):+.2f} m | min = {np.min(eta):+.2f} m | skew = {skew(eta):+.3f}")
print(f"Sub eta_sub : min = {np.min(eta_sub):+.2f} m | max = {np.max(eta_sub):+.2f} m  (Q4c, not added to eta)")
print(f"Linear u1(0):   std = {np.std(u1):.3f} m/s | max = {np.max(u1):+.2f} m/s")
print(f"Total  u(0) :   std = {np.std(u):.3f} m/s | max = {np.max(u):+.2f} m/s")

# Validity check (Lecture 6, slide 9): second-order series fails when a2/a > 1/4
i_max = np.argmax(eta1)
print(f"Largest crest: a = {eta1[i_max]:.2f} m, a2/a = {eta22[i_max]/eta1[i_max]:.2f}  (limit 0.25)")

# ============================================================
# 4. Plot around the largest crest
# ============================================================
t_peak = t[np.argmax(eta)]
t_win = (max(0, t_peak - 35), min(D, t_peak + 35))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True)
ax1.plot(t, eta1, color="#00a6d6", linewidth=1.3, linestyle="--", label=r"Linear $\eta_1$")
ax1.plot(t, eta, color="#1b263b", linewidth=1.6, label=r"$\eta_1 + \eta_{22}$")
ax1.plot(t, eta22, color="#d62728", linewidth=1.0, alpha=0.8, label=r"Super-harmonic $\eta_{22}$")
ax1.plot(t, bw["A"], color="gray", linewidth=0.8, linestyle=":", label=r"Envelope $\pm A$")
ax1.plot(t, -bw["A"], color="gray", linewidth=0.8, linestyle=":")
ax1.axhline(0, color="gray", linewidth=0.7, linestyle=":")
ax1.set_ylabel(r"$\eta(t)$ [m]")
ax1.set_title(f"Narrowband second-order wave field at x = 0, z = 0 (Hs = {Hs} m, Tp = {Tp:.2f} s, h = {h} m)")
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.plot(t, u1, color="#fdae6b", linewidth=1.3, linestyle="--", label=r"Linear $u_1(0,t)$")
ax2.plot(t, u, color="#b30000", linewidth=1.6, label=r"$u_1 + u_{22}$")
ax2.plot(t, u22, color="#54278f", linewidth=1.0, alpha=0.8, label=r"Super-harmonic $u_{22}(0,t)$")
ax2.axhline(0, color="gray", linewidth=0.7, linestyle=":")
ax2.set_xlabel("Time t [s]")
ax2.set_ylabel(r"$u(z=0, t)$ [m/s]")
ax2.set_xlim(t_win)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# Q4c: sub-harmonic bound long wave under the groups
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, eta1, color="#00a6d6", linewidth=0.6, alpha=0.6, label=r"Linear $\eta_1$")
ax.plot(t, eta_sub, color="#d62728", linewidth=1.6, label=r"Bound long wave $\eta_{sub}$")
ax.set_xlim(t_win[0] - 100, t_win[1] + 100)
ax.set_xlabel("Time t [s]"); ax.set_ylabel(r"$\eta$ [m]")
ax.set_title("Q4c: set-down under wave groups (Lecture 6, slide 19)")
ax.legend(); ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout(); plt.show()


In [ ]:
# --- Verification 1: single monochromatic wave vs. analytical Stokes 2nd order (Lecture 6, slide 6) ---
# For one component the envelope is constant (A = a), so the narrowband result must equal the Stokes solution.
T0, a0, alpha0, z_test = 10.0, 1.5, 0.7, -3.5
D_test, dt_test = 100.0, 0.25
N_test = int(round(D_test / dt_test))
df_test = 1.0 / D_test
n0 = int(round((1.0 / T0) / df_test))
w0 = 2.0 * np.pi * n0 * df_test
k0 = solve_dispersion(np.array([w0]), h, g)[0]
comp_mono = {"N_t": N_test, "dt": dt_test, "D": D_test, "t": np.arange(N_test) * dt_test,
             "df": df_test, "n_idx": np.array([n0]), "f": np.array([n0 * df_test]),
             "omega": np.array([w0]), "k": np.array([k0]), "a": np.array([a0]),
             "alpha": np.array([alpha0]), "h": h, "g": g}
lin_m = synthesize_linear(comp_mono, z=0.0)
bw_m = narrowband_bound_waves(lin_m["eta1"], 1.0 / (n0 * df_test), h, z=z_test)

psi0 = w0 * comp_mono["t"] + alpha0
eta2_stokes = (k0 * a0**2 / 4.0) * np.cosh(k0 * h) * (2.0 + np.cosh(2.0 * k0 * h)) / np.sinh(k0 * h)**3 * np.cos(2.0 * psi0)
u2_stokes = 0.75 * a0**2 * w0 * k0 * np.cosh(2.0 * k0 * (z_test + h)) / np.sinh(k0 * h)**4 * np.cos(2.0 * psi0)
print("Verification 1 - monochromatic wave vs. analytical Stokes 2nd order:")
print(f"  max |eta22 - eta2_Stokes| = {np.max(np.abs(bw_m['eta22'] - eta2_stokes)):.2e} m")
print(f"  max |u22   - u2_Stokes|   = {np.max(np.abs(bw_m['u22'] - u2_stokes)):.2e} m/s")

# --- Verification 2: worked example of Lecture 6, slide 18 (Hs = 3 m, Tp = 8 s, h = 15 m) ---
comp_ex = build_linear_components(3.0, 8.0, 15.0, D=D, dt=dt, f_min=f_min, f_max=f_max, gamma=3.3, seed=41)
lin_ex = synthesize_linear(comp_ex, z=0.0)
bw_ex = narrowband_bound_waves(lin_ex["eta1"], 8.0, 15.0)
i_ex = np.argmax(lin_ex["eta1"])
print("Verification 2 - Lecture 6, slide 18 (slide: kp*h = 1.15, G = 1.06, largest crest +22%):")
print(f"  kp*h = {bw_ex['kph']:.2f}, G = {bw_ex['G']:.2f}, largest crest {lin_ex['eta1'][i_ex]:.2f} -> "
      f"{lin_ex['eta1'][i_ex] + bw_ex['eta22'][i_ex]:.2f} m "
      f"(+{100 * bw_ex['eta22'][i_ex] / lin_ex['eta1'][i_ex]:.0f}%)  [different random record, so % differs]")

# --- Verification 3: spectral recovery (Welch PSD vs. target JONSWAP) ---
f_w, P_eta1 = welch(eta1, fs=1.0 / dt, nperseg=1024)
_, P_eta_tot = welch(eta, fs=1.0 / dt, nperseg=1024)
print(f"Verification 3 - 4*sqrt(m0) of target spectrum = {4*np.sqrt(np.sum(comp['E'] * comp['df'])):.3f} m, "
      f"4*std(eta1) = {4*np.std(eta1):.3f} m")

plt.figure(figsize=(8.5, 4.2))
plt.semilogy(comp["f"], comp["E"], color="#1b263b", linewidth=2.0, label="Target JONSWAP E(f)")
plt.semilogy(f_w, P_eta1, color="#00a6d6", linestyle="--", linewidth=1.4, label=r"Simulated $\eta_1$ (Welch PSD)")
plt.semilogy(f_w, P_eta_tot, color="#d62728", linewidth=1.4, label=r"Simulated $\eta_1 + \eta_{22}$ (Welch PSD)")
plt.xlim(0, 0.7)
plt.ylim(1e-5, 2 * np.max(comp["E"]))
plt.xlabel("Frequency f [Hz]")
plt.ylabel(r"Variance density $E(f)$ [m$^2$/Hz]")
plt.title("Verification: spectral recovery and super-harmonic energy around 2fp")
plt.grid(True, which="both", linestyle=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## Question 6 + 7 ##

In [ ]:
# Q6 - Linear kinematics from the SAME record as Q4 (comp, seed 41), as allowed by the assignment
rho = 1025.0  # [kg/m^3]

lin_bed = synthesize_linear(comp, z=-h)     # transfer omega / sinh(kh) at z = -h
u_bed = lin_bed["u1"]
dpdx_bed = -rho * lin_bed["dudt1"]          # linearised Euler: dp/dx = -rho du/dt (no current)

u_surface = lin["u1"]                       # Q6b: linear velocity at z = 0 (not the 2nd-order u)

for name, signal, unit in [
    ("Near-bed velocity", u_bed, "m/s"),
    ("Near-surface velocity", u_surface, "m/s"),
    ("Near-bed pressure gradient", dpdx_bed, "Pa/m")
]:
    print(name)
    print(f"  Mean = {np.mean(signal):.3f} {unit}")
    print(f"  Std = {np.std(signal):.3f} {unit}")
    print(f"  Maximum = {np.max(signal):.3f} {unit}")
    print(f"  Minimum = {np.min(signal):.3f} {unit}")
    print(f"  Absolute maximum = {np.max(np.abs(signal)):.3f} {unit}")


In [ ]:
# Time series
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t, u_surface, color="#e6550d", label="Near-surface velocity")
ax1.plot(t, u_bed, color="#00a6d6", label="Near-bed velocity")
ax1.axhline(0, color="gray", linestyle="--")
ax1.set_ylabel("Velocity [m/s]")
ax1.set_title("Wave-induced horizontal velocities")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)
ax2.plot(t, dpdx_bed, color="#1b263b")
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_ylabel("Pressure gradient [Pa/m]")
ax2.set_xlabel("Time [s]")
ax2.set_title("Near-bed pressure gradient")
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.set_xlim(0, 120)
plt.tight_layout()
plt.show()

In [ ]:
# Q6 - Spectral verification: sigma^2 = sum |T(f)|^2 E(f) df  (same spectrum as the record)
f, E, df = comp["f"], comp["E"], comp["df"]
omega, k = comp["omega"], comp["k"]
H_bed = omega / np.sinh(k * h)
H_surface = omega * np.cosh(k * h) / np.sinh(k * h)
H_dpdx = rho * omega**2 / np.sinh(k * h)
E_bed = H_bed**2 * E
E_surface = H_surface**2 * E
E_dpdx = H_dpdx**2 * E
for name, signal, spectrum, unit in [
    ("Near-bed velocity", u_bed, E_bed, "m/s"),
    ("Near-surface velocity", u_surface, E_surface, "m/s"),
    ("Pressure gradient", dpdx_bed, E_dpdx, "Pa/m")
]:
    sigma_time = np.std(signal)
    sigma_spectrum = np.sqrt(np.sum(spectrum * df))
    print(f"{name}: time = {sigma_time:.3f}, spectrum = {sigma_spectrum:.3f} {unit}")

plt.figure(figsize=(8, 4.5))
plt.plot(f, E / np.sum(E * df), label="Surface elevation")
plt.plot(f, E_bed / np.sum(E_bed * df), label="Near-bed velocity")
plt.plot(f, E_surface / np.sum(E_surface * df), label="Near-surface velocity")
plt.plot(f, E_dpdx / np.sum(E_dpdx * df), label="Near-bed pressure gradient")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Normalised variance density [1/Hz]")
plt.title("Comparison of normalised spectra")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Optional Q6: empirical exceedance of positive cycle maxima
def cycle_maxima(signal):
    crossings = np.where((signal[:-1] <= 0) & (signal[1:] > 0))[0] + 1
    return np.array([np.max(signal[i:j]) for i, j in zip(crossings[:-1], crossings[1:])])

plt.figure(figsize=(8, 4.5))
for name, signal in [("Near-bed velocity", u_bed), ("Near-surface velocity", u_surface), ("Pressure gradient", dpdx_bed)]:
    peaks = np.sort(cycle_maxima(signal))[::-1] / np.std(signal)
    P = np.arange(1, len(peaks)+1)/(len(peaks)+1)
    plt.semilogy(peaks, P, ".", label=name)
x = np.linspace(0, 5, 200)
plt.semilogy(x, np.exp(-x**2/2), "k--", label="Rayleigh")
plt.xlabel("Positive cycle maximum / standard deviation [-]")
plt.ylabel("Exceedance probability [-]")
plt.ylim(1e-3, 1)
plt.legend()
plt.grid(True, which="both", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Q7 - Characteristic dimensionless numbers (same Tp as Q4)
W = 60.0  # Wall width [m]
omega_p = 2*np.pi/Tp
k_p = fsolve(lambda kp: omega_p**2 - g*kp*np.tanh(kp*h), omega_p**2/g)[0]
lambda_p = 2*np.pi/k_p
print(f"Peak period = {Tp:.2f} s")
print(f"Local wavelength = {lambda_p:.2f} m")
print(f"k_p h = {k_p*h:.3f}")
print(f"k_p a = {k_p*Hs/2:.3f}")
print(f"Hs/lambda_p = {Hs/lambda_p:.3f}")
print(f"Ursell number = {Hs*lambda_p**2/h**3:.3f}")
print(f"W/lambda_p = {W/lambda_p:.3f}")
print(f"Hs/h = {Hs/h:.3f}")